In [ ]:
import pandas as pd
import zipfile
import os
import json

#Project Params

params = {}
with open("params_file.json") as param_file:
    params = json.load(param_file)


def concatenate_zips(folder_path):
    all_dataframes = []

    # Loop through every file in the directory
    for filename in os.listdir(folder_path):
        if filename.endswith(".zip"):
            file_path = os.path.join(folder_path, filename)
            #Crawl through directory a
            with zipfile.ZipFile(file_path, 'r') as z:
                for internal_file in z.namelist():
                    if internal_file.endswith(".csv"):
                        with z.open(internal_file) as f:
                            df = pd.read_csv(f)
                            df['source_file'] = filename
                            all_dataframes.append(df)
    
    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)
        return master_df
    else:
        print("Empty Dataset Directory")
        return None

FileNotFoundError: [Errno 2] No such file or directory: 'params.json'

In [ ]:
#Load main Data Table
full_dataframe = concatenate_zips(params["Dataset_Directory"] + "Project\\Flight Data")

C:\Users\cadek\AppData\Local\Temp\ipykernel_32276\3610890433.py:17: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


In [ ]:
#Load Lookup Table
airport_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\T_MASTER_CORD.csv", index_col="AIRPORT_ID")
carrier_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\Carriers.csv")

In [9]:
full_dataframe.head()

,YEAR,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,...,DIVERTED,AIR_TIME,FLIGHTS,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,source_file
0,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5225.0,10397,1039707,30397,...,0.0,33.0,1.0,164.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
1,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5115.0,11433,1143302,31295,...,0.0,89.0,1.0,651.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
2,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5414.0,11423,1142308,31423,...,0.0,87.0,1.0,533.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
3,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5284.0,12953,1295304,31703,...,0.0,103.0,1.0,722.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
4,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5341.0,10994,1099402,30994,...,0.0,83.0,1.0,641.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip


In [26]:
print(type(dest_counts['AIRPORT_ID'].iloc[0]))
print(type(airport_lookup.index[0]))

<class 'numpy.int64'>
<class 'numpy.int64'>


In [ ]:
#Find top 100 airports
dest_counts = full_dataframe['DEST_AIRPORT_ID'].value_counts().reset_index()
dest_counts.columns = ['AIRPORT_ID', 'FLIGHT_COUNT']
dest_counts.head(100)

airport_lookup.drop_duplicates()

dest_counts['AIRPORT_ID'] = dest_counts['AIRPORT_ID'].astype(int)

dest_data = pd.merge(
    dest_counts, 
    airport_lookup, 
    left_on='AIRPORT_ID', 
    right_on='AIRPORT_ID', # Use 'Code' if it's a column, or right_index=True if it's the index
    how='left'
)

dest_data.head(100)

,AIRPORT_ID,FLIGHT_COUNT,AIRPORT_SEQ_ID,AIRPORT,DISPLAY_AIRPORT_NAME,DISPLAY_AIRPORT_CITY_NAME_FULL,AIRPORT_WAC,AIRPORT_COUNTRY_NAME,AIRPORT_COUNTRY_CODE_ISO,AIRPORT_STATE_NAME,...,LATITUDE,LON_DEGREES,LON_HEMISPHERE,LON_MINUTES,LON_SECONDS,LONGITUDE,AIRPORT_START_DATE,AIRPORT_THRU_DATE,AIRPORT_IS_CLOSED,AIRPORT_IS_LATEST
0,10397,342557,1039701,ATL,Atlanta Municipal,"Atlanta, GA",34,United States,US,Georgia,...,33.640833,84.0,W,25.0,38.0,-84.427222,1/1/1950 12:00:00 AM,6/30/1971 12:00:00 AM,0,0
1,10397,342557,1039702,ATL,William B. Hartsfield Atlanta International,"Atlanta, GA",34,United States,US,Georgia,...,33.640833,84.0,W,25.0,38.0,-84.427222,7/1/1971 12:00:00 AM,9/30/2003 12:00:00 AM,0,0
2,10397,342557,1039703,ATL,Hartsfield-Jackson Atlanta International,"Atlanta, GA",34,United States,US,Georgia,...,33.640833,84.0,W,25.0,38.0,-84.427222,10/1/2003 12:00:00 AM,6/30/2011 12:00:00 AM,0,0
3,10397,342557,1039704,ATL,Hartsfield-Jackson Atlanta International,"Atlanta, GA",34,United States,US,Georgia,...,33.636667,84.0,W,25.0,41.0,-84.428056,7/1/2011 12:00:00 AM,5/31/2012 12:00:00 AM,0,0
4,10397,342557,1039705,ATL,Hartsfield-Jackson Atlanta International,"Atlanta, GA",34,United States,US,Georgia,...,33.636667,84.0,W,25.0,40.0,-84.427778,6/1/2012 12:00:00 AM,10/31/2017 12:00:00 AM,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,13303,109923,1330304,MIA,Miami International,"Miami, FL",33,United States,US,Florida,...,25.796111,80.0,W,17.0,23.0,-80.289722,7/1/2025 12:00:00 AM,NaN,0,1
96,10693,100909,1069301,BNA,Nashville International,"Nashville, TN",54,United States,US,Tennessee,...,36.126667,86.0,W,40.0,55.0,-86.681944,1/1/1950 12:00:00 AM,6/30/2011 12:00:00 AM,0,0
97,10693,100909,1069302,BNA,Nashville International,"Nashville, TN",54,United States,US,Tennessee,...,36.124444,86.0,W,40.0,41.0,-86.678056,7/1/2011 12:00:00 AM,6/30/2025 12:00:00 AM,0,0
98,10693,100909,1069303,BNA,Nashville International,"Nashville, TN",54,United States,US,Tennessee,...,36.123889,86.0,W,40.0,40.0,-86.677778,7/1/2025 12:00:00 AM,NaN,0,1
